# 02 — SQL Business Analysis

This notebook uses SQLite so the SQL analysis is reproducible without requiring a database server. The same queries are also saved as a `.sql` file.

## 1. Load raw data into SQLite

In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv('../data/telco_churn_raw.csv')
conn = sqlite3.connect('../sql/churn.db')
df.to_sql('customers', conn, if_exists='replace', index=False)
print('customers table loaded:', len(df), 'rows')

customers table loaded: 7043 rows


## 2. Overall churn

In [2]:
query = '''SELECT COUNT(*) total_customers, SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) churned_customers, ROUND(100.0*SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END)/COUNT(*),2) churn_rate_pct FROM customers;'''
pd.read_sql_query(query, conn)

,total_customers,churned_customers,churn_rate_pct
0,7043,1869,26.54


## 3. Churn by contract

In [3]:
query = '''SELECT Contract, COUNT(*) customers, ROUND(100.0*SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END)/COUNT(*),2) churn_rate_pct FROM customers GROUP BY Contract ORDER BY churn_rate_pct DESC;'''
pd.read_sql_query(query, conn)

,Contract,customers,churn_rate_pct
0,Month-to-month,3875,42.71
1,One year,1473,11.27
2,Two year,1695,2.83


## 4. Churn by internet service

In [4]:
query = '''SELECT InternetService, COUNT(*) customers, ROUND(100.0*SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END)/COUNT(*),2) churn_rate_pct FROM customers GROUP BY InternetService ORDER BY churn_rate_pct DESC;'''
pd.read_sql_query(query, conn)

,InternetService,customers,churn_rate_pct
0,Fiber optic,3096,41.89
1,DSL,2421,18.96
2,No,1526,7.40


## 5. Retention opportunity list

In [5]:
query = '''SELECT customerID, tenure, Contract, MonthlyCharges, InternetService FROM customers WHERE Contract='Month-to-month' AND tenure<6 AND Churn='No' ORDER BY MonthlyCharges DESC LIMIT 10;'''
pd.read_sql_query(query, conn)

,customerID,tenure,Contract,MonthlyCharges,InternetService
0,5760-IFJOZ,3,Month-to-month,107.95,Fiber optic
1,2081-VEYEH,3,Month-to-month,107.95,Fiber optic
2,6734-GMPVK,5,Month-to-month,105.30,Fiber optic
3,4132-KALRO,4,Month-to-month,100.85,Fiber optic
4,7379-FNIUJ,2,Month-to-month,100.20,Fiber optic
5,6645-MXQJT,2,Month-to-month,97.10,Fiber optic
6,4566-NECEV,5,Month-to-month,96.55,Fiber optic
7,1393-IMKZG,1,Month-to-month,95.85,Fiber optic
8,4929-XIHVW,2,Month-to-month,95.50,Fiber optic
9,2984-AFWNC,3,Month-to-month,95.40,Fiber optic


## 6. Data-quality SQL checks

In [6]:
print('Blank TotalCharges:')
print(pd.read_sql_query("SELECT COUNT(*) AS blank_total_charges FROM customers WHERE TRIM(TotalCharges)='' OR TotalCharges IS NULL;", conn))
print('Duplicate customer IDs:')
print(pd.read_sql_query("SELECT customerID, COUNT(*) AS cnt FROM customers GROUP BY customerID HAVING COUNT(*)>1;", conn))

Blank TotalCharges:
   blank_total_charges
0                   11
Duplicate customer IDs:
Empty DataFrame
Columns: [customerID, cnt]
Index: []


## 7. Close the connection

In [7]:
conn.close()
print('SQLite connection closed.')

SQLite connection closed.
